# empiAS — Quick Start Guide

This notebook walks through a complete empiAS analysis in **under 2 minutes**, using synthetic data so you can run it immediately without downloading any datasets.

**What you'll learn:**
1. How to structure input data for empiAS
2. How to run the empirical p-value calculation
3. How to interpret the output
4. How to visualise results (volcano plot)

**Requirements:** Install empiAS first:
```bash
pip install .          # from the repo root
# or
pip install -e ".[dev]"  # editable install with dev tools
```

## 1. Import and verify installation

In [ ]:
import numpy as np
import pandas as pd
import empias

print(f"empiAS version: {empias.__version__}")
empias.test_print()

## 2. Create synthetic expression data

empiAS expects a DataFrame where:
- **Rows** = isoforms (with a `gene` index level grouping isoforms by parent gene)
- **Columns** = MultiIndex with levels `(condition, sample)` containing TPM values

We'll simulate 2,000 isoforms from 500 genes across two conditions with 3 replicates each.
A small subset of isoforms will have a **true differential signal** injected.

In [ ]:
rng = np.random.default_rng(42)

n_genes = 500
isoforms_per_gene = 4  # each gene has 4 isoforms
n_isoforms = n_genes * isoforms_per_gene
n_differential = 80  # isoforms with a real fold-change

conditions = ["control", "treated"]
replicates = ["sample_1", "sample_2", "sample_3"]

# Build gene and isoform labels
genes = [f"GENE{g:04d}" for g in range(n_genes) for _ in range(isoforms_per_gene)]
isoforms = [f"GENE{g:04d}_iso{j+1}" for g in range(n_genes) for j in range(isoforms_per_gene)]

# Simulate baseline TPM (log-normal, mimics real RNA-seq)
baseline_tpm = rng.lognormal(mean=2.0, sigma=1.5, size=n_isoforms).clip(0.01)

# Build TPM matrix: both conditions start from same baseline + replicate noise
data = {}
for cond in conditions:
    for rep in replicates:
        noise = rng.lognormal(mean=0, sigma=0.15, size=n_isoforms)
        data[(cond, rep)] = baseline_tpm * noise

# Inject differential signal into the first n_differential isoforms
# These will have 3-8x higher expression in "treated"
fold_changes = rng.uniform(3, 8, size=n_differential)
for rep in replicates:
    noise = rng.lognormal(mean=0, sigma=0.15, size=n_differential)
    data[("treated", rep)][:n_differential] *= fold_changes * noise

# Assemble DataFrame
columns = pd.MultiIndex.from_product([conditions, replicates],
                                      names=["condition", "sample"])
df = pd.DataFrame(data, columns=columns)
df.index = pd.MultiIndex.from_arrays([genes, isoforms], names=["gene", "isoform"])

print(f"Expression matrix: {df.shape[0]} isoforms x {df.shape[1]} samples")
print(f"Injected {n_differential} truly differential isoforms (first {n_differential} rows)")
df.head(8)

## 3. Run empiAS

The main function `calculate_empirical_pvalue()` does everything in one call:
1. Builds the within-replicate null distribution
2. Computes between-condition log2 fold changes
3. Calculates local empirical p-values
4. Applies Benjamini-Hochberg FDR correction

Key parameters:
- **`area`** (default 1000): local window size — how many null-distribution neighbors to compare against
- **`cutoff`** (default 0.5): minimum |log2FC| to test; smaller changes get p=1.0
- **`tpm_threshold`** (default 1.0): minimum TPM to include an isoform
- **`n_workers`** (default -1): parallel cores (-1 = all available)

In [ ]:
from empias import calculate_empirical_pvalue

results = calculate_empirical_pvalue(
    df,
    condition1="control",
    condition2="treated",
    area=1000,
    cutoff=0.5,
    tpm_threshold=1.0,
    n_workers=-1,
)

print(f"Results: {len(results)} isoforms tested")
results.head(10)

## 4. Interpret the output

The result DataFrame has one row per tested isoform:

| Column | Meaning |
|--------|---------|
| `logFC` | log2 fold change (condition2 / condition1); positive = higher in treated |
| `tpm_mean` | Mean log2 TPM across all samples (expression level) |
| `pvalues` | empiAS empirical p-value (one-tailed) |
| `fdr` | Benjamini-Hochberg adjusted p-value |

Let's check how many significant hits we found and whether they match our injected signal.

In [ ]:
FDR_THRESHOLD = 0.05

significant = results[results["fdr"] < FDR_THRESHOLD]
print(f"Significant isoforms (FDR < {FDR_THRESHOLD}): {len(significant)}")

# Check overlap with the truly differential isoforms we injected
# The result index is a MultiIndex (gene, isoform) — extract the isoform level
injected_set = set(isoforms[:n_differential])
sig_isoform_names = significant.index.get_level_values("isoform")
true_positives = [iso for iso in sig_isoform_names if iso in injected_set]
false_positives = [iso for iso in sig_isoform_names if iso not in injected_set]

print(f"True positives:  {len(true_positives)} / {n_differential} injected")
print(f"False positives: {len(false_positives)}")
print(f"Sensitivity:     {len(true_positives)/n_differential:.1%}")
if len(significant) > 0:
    print(f"Precision:       {len(true_positives)/len(significant):.1%}")

## 5. Volcano plot

Visualise the results: log2 fold change vs. significance. Truly differential isoforms are highlighted in red.

In [ ]:
try:
    import plotly.graph_objects as go

    plot_df = results.copy()
    plot_df["neg_log10_fdr"] = -np.log10(plot_df["fdr"].clip(lower=1e-10))
    plot_df["is_true"] = plot_df.index.get_level_values("isoform").isin(injected_set)

    null_iso = plot_df[~plot_df["is_true"]]
    diff_iso = plot_df[plot_df["is_true"]]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=null_iso["logFC"], y=null_iso["neg_log10_fdr"],
        mode="markers", marker=dict(size=4, color="grey", opacity=0.4),
        name="Null isoforms",
    ))
    fig.add_trace(go.Scatter(
        x=diff_iso["logFC"], y=diff_iso["neg_log10_fdr"],
        mode="markers", marker=dict(size=6, color="red", opacity=0.7),
        name="Injected differential",
    ))
    fig.add_hline(y=-np.log10(FDR_THRESHOLD), line_dash="dash", line_color="blue",
                  annotation_text=f"FDR = {FDR_THRESHOLD}")
    fig.update_layout(
        title="empiAS Volcano Plot (synthetic data)",
        xaxis_title="log2 Fold Change (treated / control)",
        yaxis_title="-log10(FDR)",
        template="plotly_white",
        width=700, height=500,
    )
    fig.show()

except ImportError:
    import matplotlib.pyplot as plt

    plot_df = results.copy()
    plot_df["neg_log10_fdr"] = -np.log10(plot_df["fdr"].clip(lower=1e-10))
    plot_df["is_true"] = plot_df.index.get_level_values("isoform").isin(injected_set)

    fig, ax = plt.subplots(figsize=(8, 5))
    null_mask = ~plot_df["is_true"]
    ax.scatter(plot_df.loc[null_mask, "logFC"], plot_df.loc[null_mask, "neg_log10_fdr"],
               s=8, c="grey", alpha=0.4, label="Null isoforms")
    ax.scatter(plot_df.loc[~null_mask, "logFC"], plot_df.loc[~null_mask, "neg_log10_fdr"],
               s=15, c="red", alpha=0.7, label="Injected differential")
    ax.axhline(-np.log10(FDR_THRESHOLD), ls="--", c="blue", lw=0.8, label=f"FDR = {FDR_THRESHOLD}")
    ax.set_xlabel("log2 Fold Change (treated / control)")
    ax.set_ylabel("-log10(FDR)")
    ax.set_title("empiAS Volcano Plot (synthetic data)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Filter significant results

A typical workflow filters by FDR and a minimum fold-change, then exports results.

In [ ]:
# Filter: FDR < 0.05 and |logFC| > 1.0 (2-fold change)
candidates = results[(results["fdr"] < 0.05) & (results["logFC"].abs() > 1.0)]
candidates_sorted = candidates.sort_values("fdr")

print(f"Candidates passing FDR < 0.05 and |logFC| > 1.0: {len(candidates_sorted)}")
candidates_sorted.head(10)

In [ ]:
# Export to CSV
# candidates_sorted.to_csv("empias_significant_results.csv")

## 7. Using your own data

### From SALMON quant.sf files

If you have SALMON quantification output (one `quant.sf` per sample), use the `SalmonAnalysis` helper:

```python
from empias import SalmonAnalysis

sa = SalmonAnalysis()

# Load quant.sf files — sample names must follow "condition_N" format
sa.load_data(
    dfs=["path/to/control_1/quant.sf", "path/to/control_2/quant.sf",
         "path/to/treated_1/quant.sf", "path/to/treated_2/quant.sf"],
    sample_names=["control_1", "control_2", "treated_1", "treated_2"],
    f_format="tsv",
)

# Group replicates by condition (auto-detected from _N suffix)
sa.merge_replicates()

# Join with a gene-isoform mapping (Parquet with 'gene' and 'isoform' columns)
sa.gene_isoform_join(transcriptome="path/to/transcriptome.parquet")

# Build the MultiIndex DataFrame
sa.create_big_df()

# Optional: median-of-ratios normalisation (DESeq2-style)
sa.DESeq2_normalization()

# Run empiAS
results = calculate_empirical_pvalue(
    sa.big_df,
    condition1="control",
    condition2="treated",
)
```

### From any expression matrix

Any DataFrame with the right shape works — just make sure:
1. **Rows** are indexed by `(gene, isoform)` as a MultiIndex
2. **Columns** are a MultiIndex with levels `(condition, sample)`
3. Values are **TPM** (or any normalised abundance; not raw counts)